In [1]:
import json
import pandas as pd
import numpy as np

In [2]:
# ── Load raw JSON ─────────────────────────────────────────────────────────────
FOUNDATION_DATA_PATH = "FoodData_Central_foundation_food_json_2026-04-30.json"
LEGACY_DATA_PATH = "FoodData_Central_sr_legacy_food_json_2018-04.json"

In [3]:
with open(FOUNDATION_DATA_PATH) as f:
    raw = json.load(f)

# The top-level key is 'FoundationFoods'; filter out any None entries (at least
# one null placeholder exists in this dataset).
foundation_foods = [f for f in raw["FoundationFoods"] if f is not None]
print(f"Loaded {len(foundation_foods)} valid food entries")

Loaded 363 valid food entries


In [4]:
# ── Nutrient selection ────────────────────────────────────────────────────────
# Only nutrients with direct relevance to flavor profiling are kept.
# Each entry maps the exact nutrient name used in the JSON → the DataFrame
# column name that will appear in the output.

NUTRIENT_MAP = {
    # Macronutrients──────────────────────────────────────────────────────── 
    # 'Energy (Atwater General Factors)' is preferred for kcal (present in ~355
    # of 363 foods).  The flattening step below merges it with the fallback.
    # NOTE: the JSON does not contain a nutrient literally named "Energy (kcal)".
    # The fallback value lives under the plain name "Energy", which appears TWICE
    # per food (once in kJ, once in kcal) -- see flatten_food() for the unit check
    # that keeps only the kcal-denominated entry.
    "Energy (Atwater General Factors)": "energy_kcal",
    "Energy":                            "_energy_kcal_fallback",  # merged below
    "Water":                            "water_g",
    "Protein":                          "protein_g",
    "Total lipid (fat)":                "fat_total_g",
    "Carbohydrate, by difference":      "carb_g",
    "Fiber, total dietary":             "fiber_g",
    # Foundation Foods label this "Sugars, Total"; the legacy dataset instead
    # calls the exact same value "Total Sugars" and never uses the other
    # string at all -- without this fallback, every legacy row (the bulk of
    # the dataset) loses this field entirely. On the handful of foods that
    # report both names, the values match, so this merge is safe (unlike the
    # energy fallback, there is no unit ambiguity to resolve here).
    "Sugars, Total":                    "sugars_total_g",
    "Total Sugars":                     "_sugars_total_fallback",  # merged below

    # Fat subtypes (affect mouthfeel and richness) ──────────────────────────
    "Fatty acids, total saturated":     "fat_saturated_g",
    "Fatty acids, total monounsaturated": "fat_monounsat_g",
    "Fatty acids, total polyunsaturated": "fat_polyunsat_g",
    "Cholesterol":                      "cholesterol_mg",

    # Sugar subtypes (different sweetness characters) ───────────────────────
    "Glucose":                          "glucose_g",
    "Fructose":                         "fructose_g",
    "Sucrose":                          "sucrose_g",
    "Lactose":                          "lactose_g",

    # Taste-active minerals ─────────────────────────────────────────────────
    "Sodium, Na":                       "sodium_mg",       # saltiness
    "Potassium, K":                     "potassium_mg",    # subtle bitter/salty
    "Calcium, Ca":                      "calcium_mg",      # chalky / dairy
    "Magnesium, Mg":                    "magnesium_mg",    # bitter notes
    "Iron, Fe":                         "iron_mg",         # metallic (esp. red meat)
    "Zinc, Zn":                         "zinc_mg",         # metallic / savory
    "Phosphorus, P":                    "phosphorus_mg",   # savory

    # Flavor-relevant vitamins ──────────────────────────────────────────────
    "Vitamin C, total ascorbic acid":   "vitamin_c_mg",    # acidity / sourness
    "Niacin":                           "niacin_mg",       # umami precursor
    "Thiamin":                          "thiamin_mg",
    "Riboflavin":                       "riboflavin_mg",
    "Vitamin B-6":                      "vitamin_b6_mg", 
    
}

print(f"Tracking {len(NUTRIENT_MAP)} nutrient fields → {len(set(NUTRIENT_MAP.values()))} output columns")

Tracking 29 nutrient fields → 29 output columns


In [5]:
# ── Flatten to DataFrame ──────────────────────────────────────────────────────
#
# Strategy:
#   • Index = food name (from the 'description' field).
#   • For each food, iterate foodNutrients and look up each nutrient name in
#     NUTRIENT_MAP.  The 'amount' field (per-100g) is used.
#   • Nutrients absent from a food are left as NaN.
#   • Energy: if 'Energy (Atwater General Factors)' is missing, fall back to
#     'Energy (kcal)' so the energy_kcal column is as complete as possible.

def flatten_food(food: dict) -> dict:
    """Return a flat dict of selected nutrients for one food entry."""
    row = {col: np.nan for col in NUTRIENT_MAP.values()}
    row["name"] = food.get("description", "Unknown")
    row["category"] = (food.get("foodCategory") or {}).get("description", np.nan)

    for nutrient_entry in food.get("foodNutrients", []):
        if not isinstance(nutrient_entry, dict):
            continue
        nutrient_info = nutrient_entry.get("nutrient")
        if not isinstance(nutrient_info, dict):
            continue
        nutrient_name = nutrient_info.get("name", "")
        if nutrient_name in NUTRIENT_MAP:
            # "Energy" is reported twice per food (kJ and kcal) under the same
            # name -- only keep the kcal-denominated entry, otherwise whichever
            # of the two happens to appear last in the array wins (and may
            # silently overwrite a correct value with the kJ figure).
            if nutrient_name == "Energy" and nutrient_info.get("unitName") != "kcal":
                continue
            col = NUTRIENT_MAP[nutrient_name]
            row[col] = nutrient_entry.get("amount", np.nan)

    return row




In [6]:
foundation_rows = [flatten_food(f) for f in foundation_foods]
foundation_df = pd.DataFrame(foundation_rows)

# Merge energy fallback: fill any NaN in energy_kcal from the fallback column
foundation_df["energy_kcal"] = foundation_df["energy_kcal"].fillna(foundation_df["_energy_kcal_fallback"])
foundation_df.drop(columns=["_energy_kcal_fallback"], inplace=True)

# Merge sugars fallback: fill any NaN in sugars_total_g from the fallback column
foundation_df["sugars_total_g"] = foundation_df["sugars_total_g"].fillna(foundation_df["_sugars_total_fallback"])
foundation_df.drop(columns=["_sugars_total_fallback"], inplace=True)

# Set food name as the index
foundation_df.set_index("name", inplace=True)

# Re-order columns:category -> macros -> fat subtypes -> sugar subtypes -> minerals -> vitamins
COL_ORDER = [
    "category", "energy_kcal", "water_g", "protein_g", "fat_total_g", "carb_g",
    "fiber_g", "sugars_total_g",
    "fat_saturated_g", "fat_monounsat_g", "fat_polyunsat_g", "cholesterol_mg",
    "glucose_g", "fructose_g", "sucrose_g", "lactose_g",
    "sodium_mg", "potassium_mg", "calcium_mg", "magnesium_mg",
    "iron_mg", "zinc_mg", "phosphorus_mg",
    "vitamin_c_mg", "niacin_mg", "thiamin_mg", "riboflavin_mg", "vitamin_b6_mg"
]
foundation_df = foundation_df[COL_ORDER]

print(f"DataFrame shape: {foundation_df.shape}  ({foundation_df.shape[0]} foods × {foundation_df.shape[1]} nutrient columns)")
print(f"\nNaN counts per column:")
print(foundation_df.isna().sum().to_string())

DataFrame shape: (363, 28)  (363 foods × 28 nutrient columns)

NaN counts per column:
category             0
energy_kcal         42
water_g              8
protein_g           11
fat_total_g         23
carb_g              42
fiber_g            178
sugars_total_g     227
fat_saturated_g    258
fat_monounsat_g    258
fat_polyunsat_g    273
cholesterol_mg     275
glucose_g          224
fructose_g         224
sucrose_g          224
lactose_g          224
sodium_mg           30
potassium_mg        12
calcium_mg          12
magnesium_mg        12
iron_mg             12
zinc_mg             12
phosphorus_mg       12
vitamin_c_mg       252
niacin_mg          187
thiamin_mg         194
riboflavin_mg      230
vitamin_b6_mg      166


In [7]:
# 
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)
foundation_df.tail(10)

,category,energy_kcal,water_g,protein_g,fat_total_g,carb_g,fiber_g,sugars_total_g,fat_saturated_g,fat_monounsat_g,fat_polyunsat_g,cholesterol_mg,glucose_g,fructose_g,sucrose_g,lactose_g,sodium_mg,potassium_mg,calcium_mg,magnesium_mg,iron_mg,zinc_mg,phosphorus_mg,vitamin_c_mg,niacin_mg,thiamin_mg,riboflavin_mg,vitamin_b6_mg
name,,,,,,,,,,,,,,,,,,,,,,,,,,,,
"Scallops, sea, frozen, wild caught",Finfish and Shellfish Products,66.40,82.50,13.50,0.49,1.97,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,313.00,245.00,10.50,30.60,0.21,1.00,214.00,NaN,NaN,NaN,NaN,NaN
"Sea bass, Chilean, frozen, wild caught",Finfish and Shellfish Products,209.00,67.50,14.90,16.60,0.14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,109.00,236.00,6.82,16.60,0.00,0.29,152.00,NaN,NaN,NaN,NaN,NaN
"Snapper, frozen, wild caught",Finfish and Shellfish Products,89.50,77.20,20.70,0.57,0.44,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,93.10,349.00,13.90,27.50,0.20,0.36,199.00,NaN,NaN,NaN,NaN,NaN
"Snow crab, legs only, frozen",Finfish and Shellfish Products,69.20,80.30,15.50,0.29,1.11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,728.00,193.00,97.80,58.40,0.28,3.37,162.00,NaN,NaN,NaN,NaN,NaN
"Squid (calamari), frozen, tubes only",Finfish and Shellfish Products,43.90,88.90,8.81,0.55,0.93,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,272.00,9.48,10.60,10.50,0.00,0.53,63.80,NaN,NaN,NaN,NaN,NaN
"Swordfish, frozen, wild caught",Finfish and Shellfish Products,152.00,70.90,19.20,8.12,0.45,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,56.90,414.00,3.62,26.70,0.10,0.54,235.00,NaN,NaN,NaN,NaN,NaN
"Tuna, ahi or yellowfin, frozen, wild caught",Finfish and Shellfish Products,102.00,73.50,24.70,0.39,-0.10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,94.40,420.00,3.19,35.50,0.59,0.35,271.00,NaN,NaN,NaN,NaN,NaN
"Turnips, raw",Vegetables and Vegetable Products,34.00,91.00,0.95,0.12,7.27,1.92,5.09,NaN,NaN,NaN,NaN,2.73,2.09,0.26,0.00,12.80,262.00,32.70,10.40,0.00,0.16,31.30,26.80,NaN,NaN,NaN,0.12
"Watermelon, seedless, flesh only, raw",Fruits and Fruit Juices,NaN,90.90,0.87,NaN,NaN,NaN,7.20,NaN,NaN,NaN,NaN,1.47,3.25,2.48,0.00,0.00,117.00,7.86,11.40,0.02,0.10,17.10,6.46,NaN,NaN,NaN,NaN


In [8]:
print(foundation_df.columns)

Index(['category', 'energy_kcal', 'water_g', 'protein_g', 'fat_total_g',
       'carb_g', 'fiber_g', 'sugars_total_g', 'fat_saturated_g',
       'fat_monounsat_g', 'fat_polyunsat_g', 'cholesterol_mg', 'glucose_g',
       'fructose_g', 'sucrose_g', 'lactose_g', 'sodium_mg', 'potassium_mg',
       'calcium_mg', 'magnesium_mg', 'iron_mg', 'zinc_mg', 'phosphorus_mg',
       'vitamin_c_mg', 'niacin_mg', 'thiamin_mg', 'riboflavin_mg',
       'vitamin_b6_mg'],
      dtype='object')


In [9]:
with open(LEGACY_DATA_PATH) as f:
    raw = json.load(f)

legacy_foods = [f for f in raw["SRLegacyFoods"] if f is not None]
print(f"Loaded {len(legacy_foods)} valid food entries")

Loaded 7793 valid food entries


In [10]:
# ── Flatten to DataFrame ──────────────────────────────────────────────────────
# Slight modification from the more general food flattening function in that we will only be using certain food categories from the legacy dataset to avoid blowing up the size unnecessarily
LEGACY_CATEGORIES = {
    "Finfish and Shellfish Products", "Vegetables and Vegetable Products", 
    "Fruits and Fruit Juices", "Legumes and Legume Products", "Nut and Seed Products", 
    "Spices and Herbs","Fats and Oils", "Dairy and Egg Products", "Cereal Grains and Pasta", 
    "Poultry Products", "Beef Products", "Pork Products", "Lamb, Veal, and Game Products", 
    "Soups, Sauces, and Gravies", "Condiments, Sauces, and Seasonings"

}

def flatten_food_legacy(food: dict) -> dict:
    """
    Return a flat dict of selected nutrients for one food entry.
    Return an empty dict if the food is not in part of one of the desired categories
    
    """
    
    if (food.get("foodCategory") or {}).get("description", np.nan) not in LEGACY_CATEGORIES:
        return None

    row = {col: np.nan for col in NUTRIENT_MAP.values()}
    row["name"] = food.get("description", "Unknown")
    row["category"] = (food.get("foodCategory") or {}).get("description", np.nan)

    for nutrient_entry in food.get("foodNutrients", []):
        if not isinstance(nutrient_entry, dict):
            continue
        nutrient_info = nutrient_entry.get("nutrient")
        if not isinstance(nutrient_info, dict):
            continue
        nutrient_name = nutrient_info.get("name", "")
        if nutrient_name in NUTRIENT_MAP:
            # "Energy" is reported twice per food (kJ and kcal) under the same
            # name -- only keep the kcal-denominated entry, otherwise whichever
            # of the two happens to appear last in the array wins (and may
            # silently overwrite a correct value with the kJ figure).
            if nutrient_name == "Energy" and nutrient_info.get("unitName") != "kcal":
                continue
            col = NUTRIENT_MAP[nutrient_name]
            row[col] = nutrient_entry.get("amount", np.nan)

    return row





print(len(legacy_foods))

In [11]:
print(len(legacy_foods))


7793


In [12]:
#legacy_rows = [flatten_food_legacy(f) for f in legacy_foods if f is not None]
#legacy_rows = [flatten_food(f) for f in foods if f]
legacy_rows = [
    flatten_food_legacy(f) for f in legacy_foods
    if f and (f.get("foodCategory") or {}).get("description") in LEGACY_CATEGORIES
]
#legacy_rows = [r for r in (flatten_food_legacy(f) for f in legacy_foods if f) if r is not None]
legacy_df = pd.DataFrame(legacy_rows)

# Merge energy fallback: fill any NaN in energy_kcal from the fallback column
legacy_df["energy_kcal"] = legacy_df["energy_kcal"].fillna(legacy_df["_energy_kcal_fallback"])
legacy_df.drop(columns=["_energy_kcal_fallback"], inplace=True)

# Merge sugars fallback: fill any NaN in sugars_total_g from the fallback column
legacy_df["sugars_total_g"] = legacy_df["sugars_total_g"].fillna(legacy_df["_sugars_total_fallback"])
legacy_df.drop(columns=["_sugars_total_fallback"], inplace=True)

# Set food name as the index
legacy_df.set_index("name", inplace=True)

# Re-order columns:category -> macros -> fat subtypes -> sugar subtypes -> minerals -> vitamins
COL_ORDER = [
    "category", "energy_kcal", "water_g", "protein_g", "fat_total_g", "carb_g",
    "fiber_g", "sugars_total_g",
    "fat_saturated_g", "fat_monounsat_g", "fat_polyunsat_g", "cholesterol_mg",
    "glucose_g", "fructose_g", "sucrose_g", "lactose_g",
    "sodium_mg", "potassium_mg", "calcium_mg", "magnesium_mg",
    "iron_mg", "zinc_mg", "phosphorus_mg",
    "vitamin_c_mg", "niacin_mg", "thiamin_mg", "riboflavin_mg", "vitamin_b6_mg"
]
legacy_df = legacy_df[COL_ORDER]

print(f"DataFrame shape: {legacy_df.shape}  ({legacy_df.shape[0]} foods × {legacy_df.shape[1]} nutrient columns)")
print(f"\nNaN counts per column:")
print(legacy_df.isna().sum().to_string())

DataFrame shape: (5002, 28)  (5002 foods × 28 nutrient columns)

NaN counts per column:
category              0
energy_kcal           0
water_g               0
protein_g             0
fat_total_g           0
carb_g                0
fiber_g             301
sugars_total_g     1277
fat_saturated_g     143
fat_monounsat_g     221
fat_polyunsat_g     219
cholesterol_mg      108
glucose_g          4142
fructose_g         4136
sucrose_g          4148
lactose_g          4161
sodium_mg             3
potassium_mg        108
calcium_mg           11
magnesium_mg        151
iron_mg              14
zinc_mg             160
phosphorus_mg       147
vitamin_c_mg        140
niacin_mg           183
thiamin_mg          186
riboflavin_mg       162
vitamin_b6_mg       231


In [13]:
legacy_df.head()

,category,energy_kcal,water_g,protein_g,fat_total_g,carb_g,fiber_g,sugars_total_g,fat_saturated_g,fat_monounsat_g,fat_polyunsat_g,cholesterol_mg,glucose_g,fructose_g,sucrose_g,lactose_g,sodium_mg,potassium_mg,calcium_mg,magnesium_mg,iron_mg,zinc_mg,phosphorus_mg,vitamin_c_mg,niacin_mg,thiamin_mg,riboflavin_mg,vitamin_b6_mg
name,,,,,,,,,,,,,,,,,,,,,,,,,,,,
"Seaweed, Canadian Cultivated EMI-TSUNOMATA, dry",Vegetables and Vegetable Products,259.00,14.00,15.30,1.39,46.20,36.70,NaN,0.45,0.11,0.75,33.00,NaN,NaN,NaN,NaN,4330.00,2940.00,299.00,692.00,66.40,2.53,260.00,29.00,3.75,0.48,1.59,0.23
"Seaweed, Canadian Cultivated EMI-TSUNOMATA, rehydrated",Vegetables and Vegetable Products,31.00,89.60,1.86,0.17,5.62,4.50,NaN,0.05,0.01,0.09,NaN,NaN,NaN,NaN,NaN,526.00,358.00,36.00,84.00,8.07,0.31,32.00,3.50,0.46,0.06,0.19,0.03
"Potatoes, hash brown, refrigerated, unprepared",Vegetables and Vegetable Products,84.00,77.80,1.75,0.08,19.20,1.80,0.91,NaN,NaN,NaN,0.00,0.54,0.36,0.00,0.00,42.00,425.00,6.00,19.00,0.48,0.38,70.00,4.90,1.78,0.03,0.02,0.26
"Potatoes, hash brown, refrigerated, prepared, pan-fried in canola oil",Vegetables and Vegetable Products,242.00,50.60,3.24,10.30,34.00,3.60,1.16,0.81,6.57,2.86,0.00,0.61,0.45,0.00,0.00,77.00,704.00,10.00,32.00,0.74,0.63,122.00,2.70,3.19,0.03,0.05,0.36
"Sweet Potatoes, french fried, frozen as packaged, salt added in processing",Vegetables and Vegetable Products,182.00,52.00,2.16,8.92,35.60,5.70,12.90,1.16,3.70,2.86,0.00,0.81,0.57,5.41,0.00,146.00,409.00,52.00,26.00,0.77,0.38,59.00,7.50,0.70,0.09,0.09,0.18


In [14]:
usda_df = pd.concat([legacy_df, foundation_df], axis = 0, ignore_index=False)

In [15]:
usda_df.head()

,category,energy_kcal,water_g,protein_g,fat_total_g,carb_g,fiber_g,sugars_total_g,fat_saturated_g,fat_monounsat_g,fat_polyunsat_g,cholesterol_mg,glucose_g,fructose_g,sucrose_g,lactose_g,sodium_mg,potassium_mg,calcium_mg,magnesium_mg,iron_mg,zinc_mg,phosphorus_mg,vitamin_c_mg,niacin_mg,thiamin_mg,riboflavin_mg,vitamin_b6_mg
name,,,,,,,,,,,,,,,,,,,,,,,,,,,,
"Seaweed, Canadian Cultivated EMI-TSUNOMATA, dry",Vegetables and Vegetable Products,259.00,14.00,15.30,1.39,46.20,36.70,NaN,0.45,0.11,0.75,33.00,NaN,NaN,NaN,NaN,4330.00,2940.00,299.00,692.00,66.40,2.53,260.00,29.00,3.75,0.48,1.59,0.23
"Seaweed, Canadian Cultivated EMI-TSUNOMATA, rehydrated",Vegetables and Vegetable Products,31.00,89.60,1.86,0.17,5.62,4.50,NaN,0.05,0.01,0.09,NaN,NaN,NaN,NaN,NaN,526.00,358.00,36.00,84.00,8.07,0.31,32.00,3.50,0.46,0.06,0.19,0.03
"Potatoes, hash brown, refrigerated, unprepared",Vegetables and Vegetable Products,84.00,77.80,1.75,0.08,19.20,1.80,0.91,NaN,NaN,NaN,0.00,0.54,0.36,0.00,0.00,42.00,425.00,6.00,19.00,0.48,0.38,70.00,4.90,1.78,0.03,0.02,0.26
"Potatoes, hash brown, refrigerated, prepared, pan-fried in canola oil",Vegetables and Vegetable Products,242.00,50.60,3.24,10.30,34.00,3.60,1.16,0.81,6.57,2.86,0.00,0.61,0.45,0.00,0.00,77.00,704.00,10.00,32.00,0.74,0.63,122.00,2.70,3.19,0.03,0.05,0.36
"Sweet Potatoes, french fried, frozen as packaged, salt added in processing",Vegetables and Vegetable Products,182.00,52.00,2.16,8.92,35.60,5.70,12.90,1.16,3.70,2.86,0.00,0.81,0.57,5.41,0.00,146.00,409.00,52.00,26.00,0.77,0.38,59.00,7.50,0.70,0.09,0.09,0.18


In [16]:
usda_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5365 entries, Seaweed, Canadian Cultivated EMI-TSUNOMATA, dry to Watermelon, seedless, rind only, raw
Data columns (total 28 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   category         5365 non-null   object 
 1   energy_kcal      5323 non-null   float64
 2   water_g          5357 non-null   float64
 3   protein_g        5354 non-null   float64
 4   fat_total_g      5342 non-null   float64
 5   carb_g           5323 non-null   float64
 6   fiber_g          4886 non-null   float64
 7   sugars_total_g   3861 non-null   float64
 8   fat_saturated_g  4964 non-null   float64
 9   fat_monounsat_g  4886 non-null   float64
 10  fat_polyunsat_g  4873 non-null   float64
 11  cholesterol_mg   4982 non-null   float64
 12  glucose_g        999 non-null    float64
 13  fructose_g       1005 non-null   float64
 14  sucrose_g        993 non-null    float64
 15  lactose_g        980 non-null    

In [17]:
usda_df["country"] = np.nan

In [18]:
usda_df.head()

,category,energy_kcal,water_g,protein_g,fat_total_g,carb_g,fiber_g,sugars_total_g,fat_saturated_g,fat_monounsat_g,fat_polyunsat_g,cholesterol_mg,glucose_g,fructose_g,sucrose_g,lactose_g,sodium_mg,potassium_mg,calcium_mg,magnesium_mg,iron_mg,zinc_mg,phosphorus_mg,vitamin_c_mg,niacin_mg,thiamin_mg,riboflavin_mg,vitamin_b6_mg,country
name,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
"Seaweed, Canadian Cultivated EMI-TSUNOMATA, dry",Vegetables and Vegetable Products,259.00,14.00,15.30,1.39,46.20,36.70,NaN,0.45,0.11,0.75,33.00,NaN,NaN,NaN,NaN,4330.00,2940.00,299.00,692.00,66.40,2.53,260.00,29.00,3.75,0.48,1.59,0.23,NaN
"Seaweed, Canadian Cultivated EMI-TSUNOMATA, rehydrated",Vegetables and Vegetable Products,31.00,89.60,1.86,0.17,5.62,4.50,NaN,0.05,0.01,0.09,NaN,NaN,NaN,NaN,NaN,526.00,358.00,36.00,84.00,8.07,0.31,32.00,3.50,0.46,0.06,0.19,0.03,NaN
"Potatoes, hash brown, refrigerated, unprepared",Vegetables and Vegetable Products,84.00,77.80,1.75,0.08,19.20,1.80,0.91,NaN,NaN,NaN,0.00,0.54,0.36,0.00,0.00,42.00,425.00,6.00,19.00,0.48,0.38,70.00,4.90,1.78,0.03,0.02,0.26,NaN
"Potatoes, hash brown, refrigerated, prepared, pan-fried in canola oil",Vegetables and Vegetable Products,242.00,50.60,3.24,10.30,34.00,3.60,1.16,0.81,6.57,2.86,0.00,0.61,0.45,0.00,0.00,77.00,704.00,10.00,32.00,0.74,0.63,122.00,2.70,3.19,0.03,0.05,0.36,NaN
"Sweet Potatoes, french fried, frozen as packaged, salt added in processing",Vegetables and Vegetable Products,182.00,52.00,2.16,8.92,35.60,5.70,12.90,1.16,3.70,2.86,0.00,0.81,0.57,5.41,0.00,146.00,409.00,52.00,26.00,0.77,0.38,59.00,7.50,0.70,0.09,0.09,0.18,NaN


In [19]:
japan_df = pd.read_csv("japan_foods.csv", index_col="name")
unified_df = pd.concat([usda_df, japan_df])

In [20]:
unified_df.tail()

,category,energy_kcal,water_g,protein_g,fat_total_g,carb_g,fiber_g,sugars_total_g,fat_saturated_g,fat_monounsat_g,fat_polyunsat_g,cholesterol_mg,glucose_g,fructose_g,sucrose_g,lactose_g,sodium_mg,potassium_mg,calcium_mg,magnesium_mg,iron_mg,zinc_mg,phosphorus_mg,vitamin_c_mg,niacin_mg,thiamin_mg,riboflavin_mg,vitamin_b6_mg,country
name,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Fu (dried wheat gluten / yakifu),Cereal Grains and Pasta,385.00,12.00,28.50,2.70,58.70,2.30,1.40,0.40,0.40,1.10,0.00,0.30,0.30,0.80,0.00,210.00,130.00,44.00,32.00,3.40,1.60,160.00,0.00,1.10,0.10,0.08,0.07,Japan
"Anko / Tsubuan (sweet red bean paste, chunky)",Sweets,239.00,38.00,5.60,0.60,54.00,6.80,42.00,0.10,0.10,0.30,0.00,12.00,12.00,18.00,0.00,1.00,200.00,26.00,30.00,1.60,0.60,62.00,0.00,0.40,0.04,0.05,0.05,Japan
Koshian (smooth sweet red bean paste),Sweets,147.00,60.00,4.40,0.40,32.00,3.80,21.00,0.06,0.06,0.18,0.00,5.00,5.00,11.00,0.00,1.00,120.00,13.00,18.00,1.20,0.50,52.00,0.00,0.30,0.03,0.03,0.03,Japan
Mochi (fresh pounded rice cake),Sweets,235.00,44.50,4.00,0.60,50.30,0.50,0.50,0.10,0.10,0.20,0.00,0.10,0.10,0.30,0.00,2.00,52.00,4.00,11.00,0.30,1.00,56.00,0.00,0.50,0.04,0.01,0.04,Japan
Warabimochi starch,Sweets,338.00,14.70,0.10,0.10,84.80,0.10,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,2.00,10.00,10.00,2.00,0.10,0.10,5.00,0.00,0.00,0.00,0.00,0.00,Japan


In [21]:
unified_df.to_csv("starting_database.csv", index=True)
